In [ ]:
# ============================================================
# DINO-style self-supervised domain adaptation for SigLIP2 vision encoder
#
# Training data:
#   unlabeled-images/images only
#
# Method:
#   student SigLIP2 vision encoder
#   teacher SigLIP2 vision encoder = EMA copy of student
#   projection head outputs prototype logits, not class logits
#
# Training:
#   two augmented views are generated from each unlabeled image
#   student and teacher produce normalized image embeddings
#   cross-view DINO-style self-distillation loss is optimized
#   teacher and teacher projection head are updated with EMA
#   running center is updated from teacher logits
#
# Checkpoint strategy:
#   train exactly 5 epochs
#   save one checkpoint after every epoch
#   no labeled data, linear probe, kNN, or labeled validation is used
#
# ============================================================

from pathlib import Path
import copy
import random
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


# ============================================================
# 0. Config
# ============================================================

DATA_ROOT = Path("/kaggle/input/datasets/kelkalot/the-hyper-kvasir-dataset")
MODEL_NAME = "google/siglip2-base-patch16-224"
UNLABELED_IMAGE_DIR = DATA_ROOT / "unlabeled-images" / "images"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

RANDOM_STATE = 42

DINO_OUTPUT_DIR = Path("/kaggle/working/siglip2_dino_style_unlabeled_full")
DINO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DINO_MAX_UNLABELED_IMAGES = None

DINO_BATCH_SIZE = 32
DINO_NUM_WORKERS = 2
DINO_NUM_EPOCHS = 5

DINO_LR_VISION = 5e-6
DINO_LR_HEAD = 1e-4
DINO_WEIGHT_DECAY = 1e-4

UNFREEZE_LAST_VISION_LAYERS = 1

DINO_PROJ_HIDDEN_DIM = 1024
DINO_PROJ_OUT_DIM = 4096

STUDENT_TEMP = 0.10
TEACHER_TEMP = 0.04

EMA_MOMENTUM = 0.996
CENTER_MOMENTUM = 0.90

GRAD_CLIP_NORM = 1.0


# ============================================================
# 1. Seed
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_STATE)


# ============================================================
# 2. Augmentations
# ============================================================

class GaussianBlurTransform:
    def __init__(self, radius_min=0.1, radius_max=1.2, p=0.2):
        self.radius_min = radius_min
        self.radius_max = radius_max
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img
        radius = random.uniform(self.radius_min, self.radius_max)
        return img.filter(ImageFilter.GaussianBlur(radius=radius))


def build_dino_view_transform(strength="global"):
    if strength == "global":
        scale = (0.65, 1.0)
        color_jitter = transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08, hue=0.015)
        blur_p = 0.15
    elif strength == "local":
        scale = (0.45, 0.90)
        color_jitter = transforms.ColorJitter(brightness=0.18, contrast=0.18, saturation=0.12, hue=0.02)
        blur_p = 0.25
    else:
        raise ValueError(f"Unknown transform strength: {strength}")

    return transforms.Compose([
        transforms.RandomResizedCrop(size=224, scale=scale, ratio=(0.85, 1.15)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.15),
        transforms.RandomRotation(degrees=8),
        color_jitter,
        GaussianBlurTransform(radius_min=0.1, radius_max=1.2, p=blur_p),
    ])


# ============================================================
# 3. Collect unlabeled image paths
# ============================================================

def collect_image_paths(image_dir, exts, max_images=None, random_state=42):
    if not image_dir.exists():
        raise FileNotFoundError(f"Image directory not found: {image_dir}")

    paths = []
    for p in tqdm(image_dir.iterdir(), desc="Scanning unlabeled images"):
        if p.is_file() and p.suffix.lower() in exts:
            paths.append(p)

    paths = sorted(paths)

    if len(paths) == 0:
        raise ValueError(f"No images found under: {image_dir}")

    if max_images is not None and len(paths) > max_images:
        rng = np.random.default_rng(random_state)
        idx = rng.choice(len(paths), size=max_images, replace=False)
        paths = [paths[i] for i in sorted(idx)]

    print(f"Unlabeled images used for DINO-style training: {len(paths)}")
    return paths


# ============================================================
# 4. Dataset
# ============================================================

class UnlabeledDinoDataset(Dataset):
    def __init__(self, image_paths, transform1, transform2):
        self.image_paths = [str(p) for p in image_paths]
        self.transform1 = transform1
        self.transform2 = transform2

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        image = Image.open(path).convert("RGB")
        view1 = self.transform1(image)
        view2 = self.transform2(image)
        return view1, view2, path


def collate_dino_batch(batch):
    view1 = [x[0] for x in batch]
    view2 = [x[1] for x in batch]
    paths = [x[2] for x in batch]
    return view1, view2, paths


# ============================================================
# 5. DINO projection head
# ============================================================

class DinoProjectionHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x)


# ============================================================
# 6. Load SigLIP2 student / teacher
# ============================================================

def load_siglip2_student_teacher():
    from transformers import AutoProcessor, AutoModel

    device = "cuda" if torch.cuda.is_available() else "cpu"

    processor = AutoProcessor.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
    )

    student = AutoModel.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    ).to(device)

    teacher = copy.deepcopy(student).to(device)

    for p in student.parameters():
        p.requires_grad = False

    vision_layers = student.vision_model.encoder.layers
    n_layers = len(vision_layers)

    if UNFREEZE_LAST_VISION_LAYERS < 0:
        raise ValueError("UNFREEZE_LAST_VISION_LAYERS cannot be negative.")

    if UNFREEZE_LAST_VISION_LAYERS > n_layers:
        raise ValueError(
            f"UNFREEZE_LAST_VISION_LAYERS={UNFREEZE_LAST_VISION_LAYERS} "
            f"> total vision layers={n_layers}"
        )

    if UNFREEZE_LAST_VISION_LAYERS > 0:
        for layer in vision_layers[-UNFREEZE_LAST_VISION_LAYERS:]:
            for p in layer.parameters():
                p.requires_grad = True

    if hasattr(student.vision_model, "post_layernorm"):
        for p in student.vision_model.post_layernorm.parameters():
            p.requires_grad = True

    for p in teacher.parameters():
        p.requires_grad = False

    student.train()
    teacher.eval()

    trainable = sum(p.numel() for p in student.parameters() if p.requires_grad)
    total = sum(p.numel() for p in student.parameters())

    print(f"Device: {device}")
    print(f"Student trainable params: {trainable:,}")
    print(f"Student total params:     {total:,}")
    print(f"Trainable ratio:          {trainable / total:.6f}")

    return processor, student, teacher, device


# ============================================================
# 7. Encoding helpers
# ============================================================

def encode_images_with_grad(images, processor, model, device):
    inputs = processor(images=images, return_tensors="pt")

    inputs = {
        k: v.to(device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    outputs = model.get_image_features(**inputs)
    emb = outputs.pooler_output

    if emb is None:
        raise RuntimeError("SigLIP2 did not return pooler_output.")

    emb = F.normalize(emb.float(), dim=-1)
    return emb


@torch.no_grad()
def encode_images_no_grad(images, processor, model, device):
    inputs = processor(images=images, return_tensors="pt")

    inputs = {
        k: v.to(device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    outputs = model.get_image_features(**inputs)
    emb = outputs.pooler_output

    if emb is None:
        raise RuntimeError("SigLIP2 did not return pooler_output.")

    emb = F.normalize(emb.float(), dim=-1)
    return emb


# ============================================================
# 8. EMA updates
# ============================================================

@torch.no_grad()
def ema_update_teacher(student, teacher, momentum):
    student_params = dict(student.named_parameters())
    teacher_params = dict(teacher.named_parameters())

    if student_params.keys() != teacher_params.keys():
        raise RuntimeError("Student and teacher parameter structures do not match.")

    for name in teacher_params:
        teacher_params[name].data.mul_(momentum).add_(
            student_params[name].data,
            alpha=1.0 - momentum,
        )

    student_buffers = dict(student.named_buffers())
    teacher_buffers = dict(teacher.named_buffers())

    for name in teacher_buffers:
        if name in student_buffers:
            teacher_buffers[name].data.copy_(student_buffers[name].data)


@torch.no_grad()
def ema_update_head(student_head, teacher_head, momentum):
    for ps, pt in zip(student_head.parameters(), teacher_head.parameters()):
        pt.data.mul_(momentum).add_(ps.data, alpha=1.0 - momentum)


# ============================================================
# 9. DINO-style loss
# ============================================================

def dino_loss_one_direction(
    student_logits,
    teacher_logits,
    center,
    student_temp,
    teacher_temp,
):
    student_log_probs = F.log_softmax(
        student_logits / student_temp,
        dim=-1,
    )

    teacher_probs = F.softmax(
        (teacher_logits - center) / teacher_temp,
        dim=-1,
    ).detach()

    loss = -(teacher_probs * student_log_probs).sum(dim=-1).mean()
    return loss


# ============================================================
# 10. Center update
# ============================================================

@torch.no_grad()
def update_center(center, teacher_logits_list, momentum):
    batch_center = torch.cat(
        teacher_logits_list,
        dim=0,
    ).mean(dim=0, keepdim=True)

    center.mul_(momentum).add_(
        batch_center,
        alpha=1.0 - momentum,
    )

    return center


# ============================================================
# 11. Checkpoint saving
# ============================================================

def save_epoch_checkpoint(
    epoch,
    avg_loss,
    student,
    teacher,
    student_head,
    teacher_head,
    center,
    logs,
):
    checkpoint_path = (
        DINO_OUTPUT_DIR
        / f"siglip2_dino_style_epoch{epoch}.pt"
    )

    checkpoint = {
        "model_name": MODEL_NAME,
        "method": "simplified_dino_style_siglip2_unlabeled_domain_adaptation",
        "epoch": int(epoch),
        "ssl_train_loss": float(avg_loss),

        "student_state_dict": {
            k: v.detach().cpu().clone()
            for k, v in student.state_dict().items()
        },

        "teacher_state_dict": {
            k: v.detach().cpu().clone()
            for k, v in teacher.state_dict().items()
        },

        "student_head_state_dict": {
            k: v.detach().cpu().clone()
            for k, v in student_head.state_dict().items()
        },

        "teacher_head_state_dict": {
            k: v.detach().cpu().clone()
            for k, v in teacher_head.state_dict().items()
        },

        "center": center.detach().cpu().clone(),

        "unfreeze_last_vision_layers": UNFREEZE_LAST_VISION_LAYERS,
        "dino_proj_hidden_dim": DINO_PROJ_HIDDEN_DIM,
        "dino_proj_out_dim": DINO_PROJ_OUT_DIM,
        "student_temp": STUDENT_TEMP,
        "teacher_temp": TEACHER_TEMP,
        "ema_momentum": EMA_MOMENTUM,
        "center_momentum": CENTER_MOMENTUM,
        "lr_vision": DINO_LR_VISION,
        "lr_head": DINO_LR_HEAD,
        "weight_decay": DINO_WEIGHT_DECAY,
        "grad_clip_norm": GRAD_CLIP_NORM,
        "batch_size": DINO_BATCH_SIZE,
        "num_epochs": DINO_NUM_EPOCHS,
        "random_state": RANDOM_STATE,

        "logs": logs.copy(),
    }

    torch.save(
        checkpoint,
        checkpoint_path,
    )

    print(f"Saved epoch {epoch} checkpoint: {checkpoint_path}")
    return checkpoint_path


# ============================================================
# 12. Main training
# ============================================================

def run_dino_style_siglip2_unlabeled_training():
    print("========== Prepare unlabeled data ==========")

    image_paths = collect_image_paths(
        image_dir=UNLABELED_IMAGE_DIR,
        exts=IMAGE_EXTS,
        max_images=DINO_MAX_UNLABELED_IMAGES,
        random_state=RANDOM_STATE,
    )

    transform1 = build_dino_view_transform(strength="global")
    transform2 = build_dino_view_transform(strength="local")

    train_ds = UnlabeledDinoDataset(
        image_paths=image_paths,
        transform1=transform1,
        transform2=transform2,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=DINO_BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        num_workers=DINO_NUM_WORKERS,
        collate_fn=collate_dino_batch,
        pin_memory=torch.cuda.is_available(),
    )

    if len(train_loader) == 0:
        raise RuntimeError(
            "Training DataLoader contains zero batches. "
            "Reduce DINO_BATCH_SIZE or provide more unlabeled images."
        )

    print(f"Training images: {len(train_ds)}")
    print(f"Batches per epoch: {len(train_loader)}")
    print(f"Epochs: {DINO_NUM_EPOCHS}")

    print("\n========== Load student / teacher ==========")

    processor, student, teacher, device = load_siglip2_student_teacher()

    if hasattr(student.config, "vision_config"):
        embedding_dim = int(student.config.vision_config.hidden_size)
    elif hasattr(student.vision_model.config, "hidden_size"):
        embedding_dim = int(student.vision_model.config.hidden_size)
    else:
        raise RuntimeError("Could not determine SigLIP2 embedding dimension.")

    print(f"Embedding dimension: {embedding_dim}")

    student_head = DinoProjectionHead(
        input_dim=embedding_dim,
        hidden_dim=DINO_PROJ_HIDDEN_DIM,
        output_dim=DINO_PROJ_OUT_DIM,
    ).to(device)

    teacher_head = copy.deepcopy(student_head).to(device)

    for p in teacher_head.parameters():
        p.requires_grad = False

    teacher_head.eval()

    center = torch.zeros(
        1,
        DINO_PROJ_OUT_DIM,
        device=device,
    )

    student_trainable_params = [
        p for p in student.parameters()
        if p.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        [
            {
                "params": student_trainable_params,
                "lr": DINO_LR_VISION,
            },
            {
                "params": student_head.parameters(),
                "lr": DINO_LR_HEAD,
            },
        ],
        weight_decay=DINO_WEIGHT_DECAY,
    )

    logs = []
    saved_checkpoints = []

    print("\n========== Start DINO-style training ==========")

    for epoch in range(1, DINO_NUM_EPOCHS + 1):
        student.train()
        student_head.train()
        teacher.eval()
        teacher_head.eval()

        epoch_losses = []

        pbar = tqdm(
            train_loader,
            desc=f"DINO epoch {epoch}/{DINO_NUM_EPOCHS}",
        )

        for view1, view2, paths in pbar:
            optimizer.zero_grad(set_to_none=True)

            student_feat_1 = encode_images_with_grad(
                images=view1,
                processor=processor,
                model=student,
                device=device,
            )

            student_feat_2 = encode_images_with_grad(
                images=view2,
                processor=processor,
                model=student,
                device=device,
            )

            student_logits_1 = student_head(student_feat_1)
            student_logits_2 = student_head(student_feat_2)

            with torch.no_grad():
                teacher_feat_1 = encode_images_no_grad(
                    images=view1,
                    processor=processor,
                    model=teacher,
                    device=device,
                )

                teacher_feat_2 = encode_images_no_grad(
                    images=view2,
                    processor=processor,
                    model=teacher,
                    device=device,
                )

                teacher_logits_1 = teacher_head(teacher_feat_1)
                teacher_logits_2 = teacher_head(teacher_feat_2)

            loss_12 = dino_loss_one_direction(
                student_logits=student_logits_1,
                teacher_logits=teacher_logits_2,
                center=center,
                student_temp=STUDENT_TEMP,
                teacher_temp=TEACHER_TEMP,
            )

            loss_21 = dino_loss_one_direction(
                student_logits=student_logits_2,
                teacher_logits=teacher_logits_1,
                center=center,
                student_temp=STUDENT_TEMP,
                teacher_temp=TEACHER_TEMP,
            )

            loss = 0.5 * (loss_12 + loss_21)

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"DINO-style loss became NaN/Inf during epoch {epoch}."
                )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                student_trainable_params
                + list(student_head.parameters()),
                max_norm=GRAD_CLIP_NORM,
            )

            optimizer.step()

            with torch.no_grad():
                ema_update_teacher(
                    student=student,
                    teacher=teacher,
                    momentum=EMA_MOMENTUM,
                )

                ema_update_head(
                    student_head=student_head,
                    teacher_head=teacher_head,
                    momentum=EMA_MOMENTUM,
                )

                center = update_center(
                    center=center,
                    teacher_logits_list=[
                        teacher_logits_1,
                        teacher_logits_2,
                    ],
                    momentum=CENTER_MOMENTUM,
                )

            loss_value = float(loss.item())
            epoch_losses.append(loss_value)

            pbar.set_postfix(
                loss=f"{loss_value:.4f}",
                avg=f"{np.mean(epoch_losses):.4f}",
            )

        if len(epoch_losses) == 0:
            raise RuntimeError(
                f"No batches completed during epoch {epoch}."
            )

        avg_loss = float(np.mean(epoch_losses))

        print(
            f"\nEpoch {epoch}/{DINO_NUM_EPOCHS} complete | "
            f"DINO-style train loss = {avg_loss:.6f}"
        )

        log_row = {
            "epoch": epoch,
            "ssl_train_loss": avg_loss,
        }

        logs.append(log_row)

        log_df = pd.DataFrame(logs)

        log_df.to_csv(
            DINO_OUTPUT_DIR / "dino_style_training_log.csv",
            index=False,
        )

        print("\nTraining log:")
        print(log_df.to_string(index=False))

        checkpoint_path = save_epoch_checkpoint(
            epoch=epoch,
            avg_loss=avg_loss,
            student=student,
            teacher=teacher,
            student_head=student_head,
            teacher_head=teacher_head,
            center=center,
            logs=logs,
        )

        saved_checkpoints.append(checkpoint_path)

    print("\n========== Done DINO-style training ==========")
    print(f"Saved outputs to: {DINO_OUTPUT_DIR}")
    print("\nSaved checkpoints:")

    for checkpoint_path in saved_checkpoints:
        print(checkpoint_path)

    return {
        "processor": processor,
        "student": student,
        "teacher": teacher,
        "student_head": student_head,
        "teacher_head": teacher_head,
        "logs": pd.DataFrame(logs),
        "saved_checkpoints": saved_checkpoints,
    }


# ============================================================
# 13. Run
# ============================================================

dino_results = run_dino_style_siglip2_unlabeled_training()